# SmartPlant AI — Verified Data Intake & 50-Class Training
**Research prototype · not a ready classifier.**

This notebook preserves the existing Android/Streamlit 29-class model. **Nothing is merged or deployed.**

This notebook intentionally checks *two branches*: the source-evidence review branch and the archived 50-class model development branch. The current production `main` was rolled back, so cloning only `main` will not provide the 50-class scripts.

Only proceed with training after dataset licensing, expert class labels, source plant/farm grouping and independent splits have been reviewed. Use photos that you own or are authorized to train on.

## 1 · Setup research and model source branches

This downloads **source code only**, not image datasets. Re-run in a fresh Colab runtime for reproducibility. Log both Git commits and the hardware environment.

In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content")
REVIEW = ROOT / "smartplant-data-review"
MODEL = ROOT / "smartplant-model-50"
REPO = "https://github.com/dapperdude404/tinyguard-ai.git"

def run(*args, cwd=None):
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)

def clone_branch(branch, target):
    if target.exists():
        if not (target / ".git").exists():
            raise RuntimeError(f"Folder exists but is not a clone: {target}")
        print("Using existing clone", target, "(restart runtime to refresh)")
    else:
        run("git", "clone", "--depth", "1", "--branch", branch, REPO, target)
    sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=target, text=True).strip()
    print(branch, "commit:", sha)
    return sha

review_commit = clone_branch("feature/smartplant50-auto-acquire-verified-import", REVIEW)
model_commit = clone_branch("feature/smartplant-expanded-cnn-50", MODEL)

run(sys.executable, "-m", "pip", "install", "-q", "gdown", "pillow", "scikit-learn", "numpy", "requests")
import torch, torchvision
print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CUDA GPU available:", torch.cuda.is_available())

## 2 · Mount your own private image dataset

Expected: `MyDrive/SmartPlant50/data/expanded/`, with `train/c000..c049`, `val/c000..c049`, `test/c000..c049` and `groups.csv`. Every image needs a genuine source-group ID (plant or farm). No raw images are sent to GitHub.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/SmartPlant50/data/expanded")
print("Private dataset folder present:", DATA.is_dir())
print("No images are included with the GitHub repository.")

## 2A · Reuse the existing nine crop ZIPs directly from Drive — NO UPLOAD

The connected Drive already has `MyDrive/SmartPlant_Datasets/` containing nine legacy crop ZIPs. The scanner reads their **ZIP directory entries**, counts image filenames and creates a candidate mapping report in `SmartPlant50/audit_reports`. It does **not** guess labels, download external datasets, or put data into train/test folders. Optional safe staging extraction to temporary Colab disk is available, but not necessary for this inventory.

In [ ]:
SOURCE_ARCHIVES = Path("/content/drive/MyDrive/SmartPlant_Datasets")
REPORTS = Path("/content/drive/MyDrive/SmartPlant50/audit_reports")
assert SOURCE_ARCHIVES.is_dir(), "Expected your existing SmartPlant_Datasets folder"
REPORTS.mkdir(parents=True, exist_ok=True)
run(sys.executable, "scripts/scan_smartplant_drive_archives.py",
    "--source-dir", SOURCE_ARCHIVES, "--report-dir", REPORTS, cwd=REVIEW)
print("Legacy source ZIP inventory saved to", REPORTS)
print("No photos have been split, relabeled or uploaded to GitHub.")

## 2B · Create a safe 50-class folder structure and verified label plan

This uses the **existing weights-only safe checkpoint in your Drive** (`SmartPlant_Results/best_model_safe.pt`) so no model download is required. Its SHA-256 is checked against the trusted source. It also checks for duplicate crop/disease aliases, including `Corn (Maize)` vs `Maize`. The 150 class folders will initially be empty—they are *not* fabricated training data.

In [ ]:
import shutil
SAFE_CHECKPOINT = Path("/content/drive/MyDrive/SmartPlant_Results/best_model_safe.pt")
assert SAFE_CHECKPOINT.is_file(), "Missing saved trusted checkpoint in SmartPlant_Results"
run(sys.executable, "-c",
    "import sys; from smartplant.model_download import _validate; _validate(sys.argv[1]); print('Trusted safe checkpoint SHA/metadata: PASS')",
    SAFE_CHECKPOINT, cwd=MODEL)
LEGACY_LOCAL = MODEL / "models/smartplant_mobilenetv3.pt"
LEGACY_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if not LEGACY_LOCAL.exists():
    shutil.copyfile(SAFE_CHECKPOINT, LEGACY_LOCAL)
run(sys.executable, "-c",
    "import sys; from smartplant.model_download import _validate; _validate(sys.argv[1])",
    LEGACY_LOCAL, cwd=MODEL)
GENERATED_PLAN = MODEL / "data/expanded/class_plan.json"
run(sys.executable, "scripts/prepare_expanded_labels.py",
    "--checkpoint", LEGACY_LOCAL, "--output", GENERATED_PLAN, cwd=MODEL)
DATA.mkdir(parents=True, exist_ok=True)
DRIVE_PLAN = DATA / "class_plan.json"
if DRIVE_PLAN.exists() and DRIVE_PLAN.read_bytes() != GENERATED_PLAN.read_bytes():
    raise RuntimeError("Existing Drive 50-class plan differs from the trusted model: review it before proceeding")
if not DRIVE_PLAN.exists():
    shutil.copyfile(GENERATED_PLAN, DRIVE_PLAN)
for split in ("train", "val", "test"):
    for index in range(50):
        (DATA / split / f"c{index:03d}").mkdir(parents=True, exist_ok=True)
print("Created 50-class plan and empty directories in your private Drive.")
print("Important: no photographs or ground-truth labels were generated.")

## 2B.1 · Automatic verified-source discovery from Google-identified research datasets

**No manual ZIP upload.** Google Dataset Search helps discover public datasets, but the actual files and reuse terms come from the dataset publisher (for example, Mendeley Data). The importer reads our research-verified source registry, checks Mendeley's publication metadata and licence, and discovers downloadable file hashes. Running discovery does not download photos.

In [ ]:
from pathlib import Path
WORKSPACE = Path("/content/drive/MyDrive/SmartPlant50")
run(sys.executable, "scripts/acquire_smartplant50_sources.py",
    "--registry", REVIEW / "dataset_sources/50class_source_register.csv",
    "--workspace", WORKSPACE, cwd=REVIEW)
print("Source metadata report: SmartPlant50/audit_reports/external_source_receipts.json")
print("No leaf photographs have been downloaded by this discovery cell.")

## 2B.2 · Download approved provider files automatically to quarantine

**This is the acquisition step.** It downloads only supported public Mendeley files with provider-advertised SHA-256 and positively identified CC BY 4.0 licence metadata; each file must pass SHA-256 and size verification. Downloads go directly into your **private Google Drive** `SmartPlant50/source_quarantine`, never into the training dataset. A 4 GiB total budget limits network/storage use. Files that fail checks are skipped or cause the cell to fail. An open Google Images result without licensing/labels is never training ground truth.

Large datasets can exceed Colab runtime/Drive quotas. If this fails, check receipts; retries reverify cached files without corrupting them.

In [ ]:
run(sys.executable, "scripts/acquire_smartplant50_sources.py",
    "--registry", REVIEW / "dataset_sources/50class_source_register.csv",
    "--workspace", WORKSPACE, "--download",
    "--max-total-gib", "4", cwd=REVIEW)
print("Downloads complete only for provider-verified checksummed files.")
print("Unverified disease folders still need class and source-group review.")

## 2B.3 · Generate label-review suggestions from archives and new downloads

The source inventory reads ZIP files in `SmartPlant_Datasets` and `SmartPlant50/source_quarantine`. It writes a proposed mapping table. A suggested class is **not** a medically or agriculturally verified label. The review CSV begins with empty approvals and must not be filled by guessing.


In [ ]:
CANDIDATES = WORKSPACE / "audit_reports/source_folder_review_candidates.csv"
if CANDIDATES.exists():
    print("Preserving existing review CSV:", CANDIDATES)
else:
    run(sys.executable, "scripts/prepare_smartplant50_candidates.py",
        "--legacy-dir", SOURCE_ARCHIVES,
        "--workspace", WORKSPACE,
        "--plan", DRIVE_PLAN,
        "--output", CANDIDATES, cwd=REVIEW)
print("Review the suggested classes, legal permissions and real source plant/farm IDs.")
print("No class has been approved automatically.")

## 2B.4 · Verified import preflight; optional partial staging

This check decodes original image files and blocks unapproved class mappings, missing plant/farm groups, checksum changes and duplicates across different splits. By default, **dry run only**. It writes `verified_import_preflight.json`. After genuine review, running the optional staging cell creates a **new candidate folder**, not the official 50-class train/val/test dataset.

In [ ]:
assert CANDIDATES.is_file(), "Generate candidates above first"
run(sys.executable, "scripts/verified_import_smartplant50.py",
    "--review-csv", CANDIDATES, "--plan", DRIVE_PLAN,
    "--workspace", WORKSPACE, "--legacy-dir", SOURCE_ARCHIVES, cwd=REVIEW)
print("Dry-run import only; source permissions and labels remain explicit gates.")

### Optional: stage only verified photos

This cell is intentionally disabled until you have genuinely reviewed labels, rights and grouped provenance. It **never** merges the staging images into your production model. If you have completed the CSV review, change `STAGE_APPROVED = True` below.

In [ ]:
STAGE_APPROVED = False
if STAGE_APPROVED:
    run(sys.executable, "scripts/verified_import_smartplant50.py",
        "--review-csv", CANDIDATES, "--plan", DRIVE_PLAN,
        "--workspace", WORKSPACE, "--legacy-dir", SOURCE_ARCHIVES,
        "--commit", cwd=REVIEW)
else:
    print("Approved staging disabled. No training data has been changed.")

## 2C · Review new-crop camera images, if any

You can place your own photographs in `SmartPlant50/incoming_new_crops/<crop>` directly using the Google Drive phone app. These are only **unreviewed intake photos**. Qualified disease labels and source provenance are required before promoting any file into a training class. No images are automatically assigned to disease labels.

In [ ]:
INBOX = Path("/content/drive/MyDrive/SmartPlant50/incoming_new_crops")
for crop in ("Mango", "Guava", "Banana", "Rice", "Wheat", "Citrus", "Maize", "Papaya"):
    folder = INBOX / crop
    n = sum(1 for f in folder.rglob("*") if f.is_file()) if folder.is_dir() else 0
    print(f"{crop}: {n} unreviewed file(s)")
print("Even if all folders contain photos, expert labels and split provenance remain essential.")

## 3 · View gaps before attempting training

The source register begins with **21 unapproved additions**. This diagnostic step prints what remains missing. A successful inventory structure is *not* proof of ground-truth labels or dataset independence.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", cwd=REVIEW)
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA, cwd=REVIEW)

## 4 · Hard source-evidence gate

STOP until a qualified reviewer has updated the source CSV with verified permissions, class mapping, original image counts and source groups. Do not set flags to "yes" without evidence. This cell is expected to fail for the initial 21 candidate entries.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
assert DATA.is_dir(), "Missing private dataset directory"
assert (DATA / "groups.csv").is_file(), "Missing plant/farm provenance file"
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA,
    "--require-complete", cwd=REVIEW)
print("Source evidence and structural inventory passed. Full image audit still required.")

## 5 · Generate and verify the original labels

Use the original SHA-256-pinned trained checkpoint. A *class plan* is just label metadata, not training. Generate it in the model repository first; do not overwrite a differing plan stored with your data.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
run(sys.executable, "-c",
    "from smartplant.model_download import ensure_cnn_checkpoint; "
    "print(ensure_cnn_checkpoint('models/smartplant_mobilenetv3.pt'))", cwd=MODEL)
run(sys.executable, "scripts/prepare_expanded_labels.py",
    "--checkpoint", "models/smartplant_mobilenetv3.pt",
    "--output", "data/expanded/class_plan.json", cwd=MODEL)
PLAN = MODEL / "data/expanded/class_plan.json"
assert PLAN.exists(), "Class plan generation failed"
if (DATA / "class_plan.json").is_file():
    assert PLAN.read_bytes() == (DATA / "class_plan.json").read_bytes(), (
        "Drive class plan differs from generated pinned plan: review label order"
    )
print("Reviewed 29→50 label plan generated. No model was trained.")

## 6 · Strict image audit

This performs image decoding, exact/near-duplicate screening and checks that source plant/farm IDs do not cross splits. The audit must PASS before training; test the same photographic source only in one partition. A hash screen cannot guarantee complete independence.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA,
    "--require-complete", cwd=REVIEW)
run(sys.executable, "scripts/audit_expanded_dataset.py", "--dataset", DATA,
    "--plan", PLAN, "--groups-csv", DATA / "groups.csv", cwd=MODEL)
print("Strict image audit passed. Independently validate labels and provenance.")

## 7 · Fine-tune the CNN (no cloud deployment)

Even if you execute this cell out of order, it re-runs all hard gates. Trains the existing MobileNetV3-Small on all 50 classes, selecting by validation macro-F1 and evaluating the test split once. GPU runtime recommended. Output remains in your private Drive.

In [ ]:
run(sys.executable, "scripts/check_smartplant_sources.py", "--require-ready", cwd=REVIEW)
assert DATA.is_dir() and (DATA / "groups.csv").is_file(), "Dataset and provenance required"
assert (MODEL / "models/smartplant_mobilenetv3.pt").is_file(), "Verified source checkpoint missing"
assert (MODEL / "data/expanded/class_plan.json").is_file(), "Class plan missing"
run(sys.executable, "scripts/inventory_smartplant50.py", "--dataset", DATA,
    "--require-complete", cwd=REVIEW)
run(sys.executable, "scripts/audit_expanded_dataset.py", "--dataset", DATA,
    "--plan", MODEL / "data/expanded/class_plan.json", "--groups-csv", DATA / "groups.csv", cwd=MODEL)
OUTPUT = DATA.parent / "smartplant_50.pt"
run(sys.executable, "scripts/train_expanded_cnn.py", "--dataset", DATA,
    "--plan", MODEL / "data/expanded/class_plan.json",
    "--groups-csv", DATA / "groups.csv",
    "--base-checkpoint", MODEL / "models/smartplant_mobilenetv3.pt",
    "--output", OUTPUT, "--seed", "42", "--epochs", "25",
    "--batch-size", "32", "--patience", "6", "--lr", "0.0001", cwd=MODEL)
print("Research checkpoint in your private Drive:", OUTPUT)
print("Review its .metrics.json and .confusion.csv before considering any release.")

## 8 · Independent acceptance before Android/Streamlit updates

A trained model still needs external field photos, crop-mismatch checks (including the potato→strawberry failure case), a separate non-leaf rejection evaluation (fabric photo), subgroup error analysis, real-image PyTorch/ONNX parity, Android camera and airplane-mode testing.

**Never use synthetic 50-class smoke weights as a classifier.** There is no automatic GitHub merge, APK release or cloud deployment in this notebook.